In [ ]:
# Cell 1: Install Kaggle API
!pip install kaggle -q
!mkdir -p scripts models datasets


In [ ]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/access_token", "w") as f:
    f.write("access_token")

os.chmod("/root/.kaggle/access_token", 0o600)

In [ ]:
import os

os.environ["KAGGLE_USERNAME"] = "username"
os.environ["KAGGLE_KEY"] = "api_key"

In [ ]:
!python scripts/fetch_datasets.py --check-auth

19:12:37 [INFO] Found Kaggle credentials in environment variables.


In [ ]:
# Cell 3: Download dataset
# Option A: Real vs Fake (faster to download and train)
!python scripts/fetch_datasets.py --dataset real-vs-fake --dest datasets/real_vs_fake

# OR Option B: CelebA-Spoof
# !python scripts/fetch_datasets.py --dataset celeba-spoof --dest datasets/celeba_spoof


19:15:23 [INFO] Dataset: real-vs-fake (Real vs Fake Anti-Spoofing Video Classification)
19:15:23 [INFO] Target directory: datasets/real_vs_fake
19:15:23 [INFO] Found Kaggle credentials in environment variables.
19:15:23 [INFO] Starting download for Kaggle dataset: trainingdatapro/real-vs-fake-anti-spoofing-video-classification -> datasets/real_vs_fake
19:15:23 [INFO] Kaggle authentication successful.
19:15:23 [INFO] Downloading dataset archive (this may take time depending on size)...
Dataset URL: https://www.kaggle.com/datasets/trainingdatapro/real-vs-fake-anti-spoofing-video-classification
Resuming from 0 bytes (3262405767 bytes left)...
100% 3.04G/3.04G [01:21<00:00, 40.1MB/s]

19:17:04 [INFO] Download completed successfully!
19:17:04 [INFO] Ready! You can now run training with:
19:17:04 [INFO]   python scripts/train_antispoof.py --data-dir datasets/real_vs_fake


In [ ]:
# Cell 4: Preprocess
# If you downloaded videos (Real vs Fake / Replay-Attack):
!python scripts/create_dataset.py --mode videos --input-dir datasets/real_vs_fake --output-dir datasets/processed_pad --frame-interval 8

# If you downloaded CelebA-Spoof (extract a fast 5,000-sample balanced subset):
# !python scripts/create_dataset.py --mode celeba-subset --input-dir datasets/celeba_spoof --output-dir datasets/processed_pad --max-per-class 2500


19:29:10 [INFO] Found metadata CSV: datasets/real_vs_fake/real_and_fake.csv. Inspecting...
19:29:10 [INFO] Successfully matched 160 media items from real_and_fake.csv
19:29:10 [INFO] Discovered 160 video(s) to process. Extracting face crops...
19:29:10 [INFO] OpenCV CascadeClassifier unavailable in current environment. Using smart portrait face cropping.
19:30:06 [INFO] Progress: 20/160 videos processed (294 real crops, 300 spoof crops)
19:31:22 [INFO] Progress: 40/160 videos processed (583 real crops, 600 spoof crops)
19:32:05 [INFO] Progress: 60/160 videos processed (843 real crops, 894 spoof crops)
19:33:07 [INFO] Progress: 80/160 videos processed (1139 real crops, 1194 spoof crops)
19:33:33 [INFO] Progress: 100/160 videos processed (1417 real crops, 1492 spoof crops)
19:34:43 [INFO] Progress: 120/160 videos processed (1702 real crops, 1778 spoof crops)
19:35:50 [INFO] Progress: 140/160 videos processed (1988 real crops, 2076 spoof crops)
19:37:00 [INFO] Progress: 160/160 videos pro

In [ ]:
# Cell 5: Train ensemble on GPU
!python scripts/train_antispoof.py \
    --data-dir datasets/processed_pad \
    --model ensemble \
    --epochs 12 \
    --batch-size 32 \
    --lr 0.0003 \
    --output models/antispoof_fullmodels.pkl


19:37:15 [INFO] Using device: cuda
19:37:15 [INFO] Dataset loaded: 3206 train samples, 1458 val samples.
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth
100% 13.6M/13.6M [00:00<00:00, 143MB/s]
19:37:15 [INFO] Training MobileNetV2 on cuda for 12 epochs...
19:37:33 [INFO] [MobileNetV2] Epoch 01/12 (17.4s) | Train Loss: 0.2310, Acc: 89.9% | Val Acc: 84.8%, APCER: 13.9%, BPCER: 16.4%, ACER: 15.17%
19:37:49 [INFO] [MobileNetV2] Epoch 02/12 (16.0s) | Train Loss: 0.0495, Acc: 98.5% | Val Acc: 84.5%, APCER: 19.5%, BPCER: 11.5%, ACER: 15.47%
19:38:05 [INFO] [MobileNetV2] Epoch 03/12 (16.4s) | Train Loss: 0.0261, Acc: 99.1% | Val Acc: 88.8%, APCER: 15.4%, BPCER: 7.0%, ACER: 11.22%
19:38:21 [INFO] [MobileNetV2] Epoch 04/12 (15.9s) | Train Loss: 0.0373, Acc: 98.9% | Val Acc: 85.1%, APCER: 19.1%, BPCER: 10.6%, ACER: 14.85%
19:38:37 [INFO] [MobileNetV2] Epoch 05/12 (15.9s) | Train Loss: 0.0153, Acc: 99.6% | 

In [ ]:
from google.colab import files
files.download('models/antispoof_fullmodels.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>